# Corrected Comparison of Graph-Coloring Searches

This notebook analyses a controlled study (`kind: study`) of complete Grover
searches for graph coloring. Every Logique synthesis method and every
Saha–Belletti baseline is verified against the same predicate, its
multi-controlled gates are expanded by named decompositions, and the
preparation, oracle and diffusion operator are priced separately at the
logical, Toffoli, fault-tolerant and routed levels. The protocol is described
in `docs/research/experimental_protocol.md`.

The study is run separately, because the nine-graph evaluation takes hours, and
the notebook is pointed at its output folder:

```bash
uv run --no-sync logique benchmark run --config configs/corrected_comparison.json
LOGIQUE_STUDY_RUN=dump/MMDDHHMMSS-corrected-study \
  uv run --no-sync logique notebook run experiments/studies/corrected_comparison.ipynb --offline
```

The sections cover the coverage of the run, the solution counts and iteration
counts, the T-count of the complete search, the comparison with the strongest
baseline, the effect of the three regimes, the width by qubit role, the
trade-off between width and T-count, measurement-assisted pricing and routed
metrics.

## 1. Load the study

`LOGIQUE_STUDY_RUN` names the study folder; the notebook never selects a run
implicitly. Tables and figures are written to the notebook's own output folder.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.ticker import LogLocator

from logique.benchmarks import analysis
from logique.paths import new_dump_directory

ROOT = Path(os.environ.get("LOGIQUE_WORKSPACE", Path.cwd())).resolve()
if "LOGIQUE_STUDY_RUN" not in os.environ:
    raise RuntimeError("Set LOGIQUE_STUDY_RUN to the folder of a completed study")
RUN = (ROOT / os.environ["LOGIQUE_STUDY_RUN"]).resolve()
if "LOGIQUE_OUTPUT" in os.environ:
    OUT = Path(os.environ["LOGIQUE_OUTPUT"]).resolve()
else:
    OUT = new_dump_directory("corrected-comparison", root=ROOT / "dump")
FIGURES = OUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
rows, manifest = analysis.load_study(RUN)

# Reference palette of the data-visualization guidelines.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, SECONDARY, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "font.size": 9,
        "axes.edgecolor": AXIS,
        "axes.labelcolor": SECONDARY,
        "xtick.color": SECONDARY,
        "ytick.color": SECONDARY,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "axes.axisbelow": True,
    }
)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
GRAPHS = [Path(p).stem for p in manifest["settings"]["inputs"]]
LOGIQUE = [m for m in manifest["settings"]["methods"] if not m.startswith("sb_")]
BASELINES = [m for m in manifest["settings"]["methods"] if m.startswith("sb_")]
POLICIES = manifest["settings"]["mcx_policies"]


def show_figure(figure, stem: str) -> None:
    """Save a figure as PNG and SVG, then display it."""
    figure.savefig(FIGURES / f"{stem}.svg", bbox_inches="tight")
    figure.savefig(FIGURES / f"{stem}.png", dpi=140, bbox_inches="tight")
    display(Image(filename=str(FIGURES / f"{stem}.png")))
    plt.close(figure)


display(
    Markdown(
        f"Study `{RUN.name}`: {len(manifest['cases'])} cases, {len(GRAPHS)} graphs, "
        f"{len(LOGIQUE)} Logique methods, {len(BASELINES)} baselines, "
        f"MCX policies {', '.join(POLICIES)}; checkout "
        f"`{manifest['checkout'].get('revision', manifest['checkout'])}`."
    )
)

## 2. Coverage and verification

Every case records its oracle form (direct kick-back or compute–phase–uncompute)
and its verification. Instances with at most 12 data qubits are verified
exhaustively; larger ones on explicit assignments that include proper
colorings as positive witnesses, so the marking branch of every oracle is
exercised. Cases that exceeded a limit appear with their status.

In [ ]:
cases = pd.DataFrame(
    [
        {
            "benchmark": case["benchmark"],
            "method": case["method"],
            "regime": case["regime"],
            "status": case["status"],
            "form": case.get("oracle_form"),
            "validation": case.get("validation", {}).get("status"),
            "witnesses": case.get("validation", {}).get("coverage", {}).get("witnesses"),
            "marked": case.get("validation", {}).get("marked_assignments"),
        }
        for case in manifest["cases"]
    ]
)
display(
    cases.pivot_table(
        index="benchmark",
        columns="status",
        values="method",
        aggfunc="count",
        fill_value=0,
    ).reindex(GRAPHS)
)
display(
    cases.groupby(["benchmark", "validation", "form"], dropna=False)
    .size()
    .rename("cases")
    .reset_index()
    .set_index("benchmark")
    .loc[[g for g in GRAPHS if g in set(cases.benchmark)]]
)
assert set(cases.status) <= {"complete", "timeout"}, "the study contains failed cases"

## 3. Domains, solutions and iteration counts

The prepared domain holds $N = 2^{c|V|}$ assignments under R1 and $k^{|V|}$
under R2 and R3, and $M$ of them are proper colorings. The optimal iteration
count is $\lfloor \pi / (4\theta) \rfloor$ with $\sin^2\theta = M/N$. Graphs
whose solutions could not be counted have no complete-search rows.

In [ ]:
costs = analysis.search_costs(rows)
domains = (
    rows[rows.component == "search"]
    .drop_duplicates(["benchmark", "regime"])
    .set_index(["benchmark", "regime"])[["domain", "solutions", "iterations", "success"]]
)
domains["initial fraction"] = domains.solutions / domains.domain.astype(float)
display(domains.reindex(GRAPHS, level=0))
uncounted = sorted(set(GRAPHS) - set(costs.benchmark))
if uncounted:
    display(Markdown(f"No complete-search rows (solutions not counted): {', '.join(uncounted)}."))

## 4. T-count of the complete search

The table gives, for every method, the total T-count of the complete search at
the optimal iteration count under its best MCX policy, including synthesized
rotations under the whole-algorithm error budget. The policy that attains it
follows in the second table.

In [ ]:
best = costs.loc[costs.groupby(["benchmark", "regime", "method"]).t_count.idxmin()]
order = [*LOGIQUE, *BASELINES]
for regime in manifest["settings"]["regimes"]:
    table = best[best.regime == regime].pivot(index="benchmark", columns="method", values="t_count")
    display(Markdown(f"**{regime}**: T-count of the complete search, best policy"))
    display(table.reindex(index=[g for g in GRAPHS if g in table.index], columns=order).round(0).astype("Int64"))
policies = best.pivot_table(
    index=["benchmark", "regime"], columns="method", values="mcx_policy", aggfunc="first"
)
display(Markdown("Policy attaining the minimum"))
display(policies.reindex(columns=order))
best.to_csv(OUT / "best_policy_search_costs.csv", index=False)

## 5. Comparison with the strongest baseline

Each Logique method is compared on every graph and regime with the strongest
of the four baselines. With matched policies, both sides use the same MCX
decomposition; with best policies, each side takes its cheapest one. A ratio
below 1 means the Logique method needs fewer resources. Outcomes within 1% are
ties; the geometric mean summarizes the ratios across graphs.

In [ ]:
comparisons = {}
for metric in ("t_count", "qubits_total"):
    for matched in (True, False):
        table = analysis.compare_with_baselines(costs, metric, matched_policy=matched)
        label = f"{metric}, {'matched' if matched else 'best'} policies"
        comparisons[label] = table
        table.to_csv(OUT / f"comparison_{metric}_{'matched' if matched else 'best'}.csv", index=False)
summary = pd.concat(
    {label: analysis.summarize_outcomes(table) for label, table in comparisons.items()},
    names=["comparison"],
)
display(summary.round(3))
summary.to_csv(OUT / "comparison_summary.csv")

The heatmaps show the T-count ratio per graph with best policies on both
sides. Blue cells are cheaper than the strongest baseline and red cells more
expensive; the color saturates at a factor of 16, and the printed value is the
exact ratio.

In [ ]:
diverging = LinearSegmentedColormap.from_list("ratio", ["#2a78d6", "#f0efec", "#e34948"])
table = comparisons["t_count, best policies"]
regimes = manifest["settings"]["regimes"]
figure, axes = plt.subplots(1, len(regimes), figsize=(4.2 * len(regimes), 0.42 * len(LOGIQUE) + 1.6), sharey=True)
for ax, regime in zip(np.atleast_1d(axes), regimes):
    grid = table[table.regime == regime].pivot(index="method", columns="benchmark", values="ratio")
    grid = grid.reindex(index=LOGIQUE, columns=[g for g in GRAPHS if g in grid.columns])
    values = np.log2(grid.to_numpy(dtype=float))
    ax.imshow(values, cmap=diverging, norm=TwoSlopeNorm(0, -4, 4), aspect="auto")
    for (i, j), ratio in np.ndenumerate(grid.to_numpy(dtype=float)):
        if np.isfinite(ratio):
            text = f"{ratio:.2f}" if ratio < 10 else f"{ratio:.0f}"
            ax.text(j, i, text, ha="center", va="center", fontsize=7, color=INK)
    ax.set_xticks(range(grid.shape[1]), grid.columns, rotation=45, ha="right")
    ax.set_yticks(range(grid.shape[0]), grid.index)
    ax.set_title(regime, color=INK)
    ax.grid(False)
figure.suptitle("T-count relative to the strongest baseline (best policies)", color=INK, y=1.02)
show_figure(figure, "t-count-ratio-heatmap")

### Per-iteration comparison

Graphs whose proper colorings could not be counted have no complete-search rows.
Every method shares the iteration count within a regime, so the cost of one
iteration (oracle and diffusion operator, with the iteration's rotations sharing
the error budget) ranks the methods on every graph, including those graphs.

In [ ]:
per_iteration = analysis.iteration_costs(
    rows, budget=manifest["settings"].get("rotation_budget", 1e-3)
)
iteration_comparisons = {}
for metric in ("t_count", "qubits_total"):
    for matched in (True, False):
        table = analysis.compare_with_baselines(per_iteration, metric, matched_policy=matched)
        label = f"{metric} per iteration, {'matched' if matched else 'best'} policies"
        iteration_comparisons[label] = table
        table.to_csv(OUT / f"comparison_iteration_{metric}_{'matched' if matched else 'best'}.csv", index=False)
iteration_summary = pd.concat(
    {label: analysis.summarize_outcomes(table) for label, table in iteration_comparisons.items()},
    names=["comparison"],
)
display(iteration_summary.round(3))
iteration_summary.to_csv(OUT / "comparison_iteration_summary.csv")
ratios = iteration_comparisons["t_count per iteration, best policies"]
display(Markdown(f"**{regimes[-1]}**: T-count per iteration relative to the strongest baseline, best policies"))
display(
    ratios[ratios.regime == regimes[-1]]
    .pivot(index="benchmark", columns="method", values="ratio")
    .reindex(index=[g for g in GRAPHS if g in set(ratios.benchmark)], columns=LOGIQUE)
    .round(3)
)

## 6. Preparation and domain checks

Comparing R1 with R2 isolates the valid-color preparation (with its rotation
cost) from the Hadamard preparation, and comparing R2 with R3 isolates the
removal of the redundant domain checks. When the number of colors is a power of
two, the three regimes coincide. The chart indexes the T-count of R2 and R3 to
that of R1, for `xag` and for the strongest baseline; points below the line are
cheaper than the Hadamard preparation with the complete predicate.

In [ ]:
regime_table = best.pivot_table(index=["benchmark", "method"], columns="regime", values="t_count")
if {"R1", "R2"} <= set(regime_table.columns):
    regime_table["R2 / R1"] = regime_table["R2"] / regime_table["R1"]
if {"R2", "R3"} <= set(regime_table.columns):
    regime_table["R3 / R2"] = regime_table["R3"] / regime_table["R2"]
display(regime_table.round(3))

strongest = (
    best[best.method.isin(BASELINES)]
    .groupby(["benchmark", "regime"])
    .t_count.min()
    .rename("strongest baseline")
)
xag = best[best.method == "xag"].set_index(["benchmark", "regime"]).t_count.rename("xag")
paired = pd.concat([xag, strongest], axis=1).reset_index()
graphs = [g for g in GRAPHS if g in set(paired.benchmark)]
figure, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, column in zip(axes, ["xag", "strongest baseline"]):
    base = paired[paired.regime == regimes[0]].set_index("benchmark")[column].reindex(graphs)
    for r, regime in enumerate(regimes[1:], start=1):
        values = paired[paired.regime == regime].set_index("benchmark")[column].reindex(graphs)
        ax.scatter(
            np.arange(len(graphs)) + (r - 1.5) * 0.16,
            values / base,
            s=40,
            color=SERIES[r],
            edgecolor="white",
            linewidth=0.8,
            label=f"{regime} / {regimes[0]}",
            zorder=2,
        )
    ax.axhline(1, color=AXIS, linewidth=1, zorder=1)
    ax.set_yscale("log")
    ax.yaxis.set_major_locator(LogLocator(subs=(1.0, 2.0, 5.0)))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.yaxis.set_minor_formatter(plt.NullFormatter())
    ax.set_xticks(range(len(graphs)), graphs, rotation=45, ha="right")
    ax.set_title(column, color=INK)
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel(f"T-count relative to {regimes[0]}")
handles, names = axes[0].get_legend_handles_labels()
figure.legend(handles, names, frameon=False, ncol=2, loc="lower center")
figure.tight_layout(rect=(0, 0.07, 1, 1))
show_figure(figure, "regimes")

## 7. Width by qubit role

The width of every circuit is broken down into data qubits, the output
(kick-back) qubit, the oracle's workspace (intermediate ANDs of a Logique
oracle, comparator and check ancillas of a baseline) and the ancillas that the
MCX decomposition appends. The chart uses regime R3 and the two-clean-ancilla
construction of Khattar and Gidney; the CSV file holds every policy and regime.

In [ ]:
widths = analysis.width_breakdown(rows)
widths.to_csv(OUT / "width_breakdown.csv", index=False)
policy = "2_clean_kg24" if "2_clean_kg24" in POLICIES else POLICIES[0]
shown = widths[(widths.regime == regimes[-1]) & (widths.mcx_policy == policy)]
labels = {
    "data_qubits": "data",
    "output_qubits": "output",
    "oracle_workspace": "oracle workspace",
    "mcx_ancillas": "MCX ancillas",
}
present = [g for g in GRAPHS if g in set(shown.benchmark)]
columns = 3
rows_n = int(np.ceil(len(present) / columns))
figure, axes = plt.subplots(rows_n, columns, figsize=(11, 2.6 * rows_n), squeeze=False)
for ax, graph in zip(axes.flat, present):
    part = shown[shown.benchmark == graph].set_index("method").reindex(order).dropna(how="all")
    left = np.zeros(len(part))
    for color, (role, label) in zip(SERIES, labels.items()):
        ax.barh(part.index, part[role], left=left, color=color, edgecolor="white", linewidth=1, height=0.7, label=label)
        left += part[role].to_numpy()
    ax.invert_yaxis()
    ax.set_title(graph, color=INK)
    ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", labelsize=7)
for ax in axes.flat[len(present):]:
    ax.axis("off")
handles, names = axes.flat[0].get_legend_handles_labels()
figure.legend(handles, names, frameon=False, ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.02))
figure.suptitle(f"Qubits by role ({regimes[-1]}, {policy})", color=INK)
figure.tight_layout(rect=(0, 0.04, 1, 0.97))
show_figure(figure, "width-by-role")

## 8. Trade-off between width and T-count

Each point is one method under one MCX policy in the last regime (R3). Colors separate
`xag`, the other Logique methods and the baselines; marker shapes give the
policy. The gray line joins the Pareto-optimal points, which no other point
improves in both width and T-count.

In [ ]:
groups = {
    "xag": (lambda m: m == "xag", SERIES[0]),
    "other Logique methods": (lambda m: m in LOGIQUE and m != "xag", SERIES[1]),
    "baselines": (lambda m: m in BASELINES, SERIES[2]),
}
markers = dict(zip(POLICIES, ["o", "s", "^", "D", "v", "P", "X"]))
points = costs[costs.regime == regimes[-1]]
present = [g for g in GRAPHS if g in set(points.benchmark)]
rows_n = int(np.ceil(len(present) / 3))
figure, axes = plt.subplots(rows_n, 3, figsize=(11, 3.1 * rows_n), squeeze=False)
fronts = []
for ax, graph in zip(axes.flat, present):
    part = points[points.benchmark == graph]
    front = analysis.pareto_front(part, "qubits_total", "t_count")
    fronts.append(front)
    # Rotation-heavy constructions reach orders of magnitude more T gates;
    # the panel shows the region within a factor of 30 of the cheapest point.
    ceiling = 30 * part.t_count.min()
    ax.step(front.qubits_total, front.t_count, where="post", color=MUTED, linewidth=1.2, zorder=1)
    for name, (member, color) in groups.items():
        for policy_name, marker in markers.items():
            chosen = part[part.method.map(member) & (part.mcx_policy == policy_name)]
            chosen = chosen[chosen.t_count <= ceiling]
            ax.scatter(chosen.qubits_total, chosen.t_count, s=34, marker=marker, color=color, edgecolor="white", linewidth=0.8, zorder=2)
    hidden = int((part.t_count > ceiling).sum())
    if hidden:
        ax.text(0.98, 0.96, f"{hidden} points above", transform=ax.transAxes, ha="right", va="top", color=MUTED, fontsize=7)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(top=ceiling)
    for axis in (ax.xaxis, ax.yaxis):
        axis.set_major_locator(LogLocator(subs=(1.0, 2.0, 5.0)))
        axis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
        axis.set_minor_formatter(plt.NullFormatter())
    ax.set_title(graph, color=INK)
for ax in axes.flat[len(present):]:
    ax.axis("off")
for ax in axes[:, 0]:
    ax.set_ylabel("T gates, complete search")
for ax in axes[-1, :]:
    ax.set_xlabel("qubits")
from matplotlib.lines import Line2D

colors = [Line2D([], [], marker="o", linestyle="", color=color, label=name) for name, (_, color) in groups.items()]
shapes = [Line2D([], [], marker=marker, linestyle="", color=SECONDARY, label=name) for name, marker in markers.items()]
figure.legend(handles=colors, frameon=False, ncol=3, loc="lower left", bbox_to_anchor=(0.02, -0.05))
figure.legend(handles=shapes, frameon=False, ncol=len(shapes), loc="lower right", bbox_to_anchor=(0.98, -0.05), fontsize=8)
figure.tight_layout(rect=(0, 0.05, 1, 1))
show_figure(figure, "pareto")
pd.concat(fronts).to_csv(OUT / "pareto_fronts.csv", index=False)
display(pd.concat(fronts)[["benchmark", "method", "mcx_policy", "qubits_total", "t_count"]].set_index("benchmark"))

## 9. Measurement-assisted pricing

Temporary logical-AND pairs can be uncomputed by measurement and a classically
controlled CZ, which removes the T gates of every uncomputation. These rows
are reported separately because they rely on mid-circuit measurement and
feedforward. The table gives the ratio of measurement-assisted to static
T-count for the complete search, per method and policy, in the last regime.

In [ ]:
assisted = analysis.search_costs(rows, "fault-tolerant, measurement-assisted")
if assisted.empty:
    display(Markdown("The study did not request measurement-assisted pricing."))
else:
    keys = ["benchmark", "regime", "method", "mcx_policy", "iterations"]
    both = costs.merge(assisted, on=keys, suffixes=("", "_assisted"))
    both["assisted / static"] = both.t_count_assisted / both.t_count
    view = both[both.regime == regimes[-1]].pivot_table(
        index=["benchmark", "mcx_policy"], columns="method", values="assisted / static"
    )
    display(view.reindex(columns=order).round(3))
    both.to_csv(OUT / "measurement_assisted.csv", index=False)

## 10. Routed metrics

One complete iteration is routed on the `FakeTorino` device model with five
fixed seeds. The table gives the median routed two-qubit count under the best
policy per method in the last regime, and the second table counts the
iterations that were not routed, with the reason.

In [ ]:
routed = rows[rows.component == "iteration"].copy()
if routed.empty:
    display(Markdown("The study did not request routed metrics."))
else:
    done = routed[routed.routed == True]  # noqa: E712
    best_routed = done.loc[done.groupby(["benchmark", "regime", "method"]).two_qubit_count.idxmin()]
    view = best_routed[best_routed.regime == regimes[-1]].pivot(
        index="benchmark", columns="method", values="two_qubit_count"
    )
    display(view.reindex(index=[g for g in GRAPHS if g in view.index], columns=order).round(0).astype("Int64"))
    skipped = routed[routed.routed != True]  # noqa: E712
    if not skipped.empty:
        reasons = skipped.reason.str.replace(r"\d+", "N", regex=True)
        display(skipped.assign(reason=reasons).groupby(["benchmark", "reason"]).size().rename("iterations").to_frame())
    best_routed.to_csv(OUT / "routed_best_policy.csv", index=False)

## 11. Record the analysis

The analysis records the study it read and the versions it ran with.

In [ ]:
from logique.benchmarks.provenance import checkout_revision, environment

record = {
    "study": str(RUN),
    "study_run_id": manifest["run_id"],
    "study_checkout": manifest["checkout"],
    "versions": environment(),
    "checkout": checkout_revision(ROOT),
}
(OUT / "analysis_manifest.json").write_text(json.dumps(record, indent=2, default=str) + "\n")
for path in sorted(OUT.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUT))